# 10 · Reconcile and publish a Silver release

Run this notebook from the `silver/` folder with the existing project kernel. It selects and verifies Bronze manifests, then rebuilds the required prior stages in memory. Reuse the same `silver_settings.json` to review a consistent selection across notebooks.

In [1]:
from IPython.display import display
import pandas as pd

from silver_notebook import build, frame, input_frame, settings_from_file

model = build("validate", settings_from_file())
display(input_frame(model))

,source,manifest
0,erpnext,batch/erpnext/manual__2026-11-21T15%3A27%3A44....
1,dealer-sql,batch/dealer-sql/manual__2026-11-21T15%3A28%3A...
2,parent-pos,batch/parent-pos/manual__2026-11-21T15%3A27%3A...
3,dealer_export,files/otc-inbound/dealer/exports/dealer-export...
4,parent_finance,files/otc-inbound/parent/finance/dealer-order-...
5,dealer_finance,files/otc-inbound/dealer/finance/dealer-order-...
6,mdm,files/otc-inbound/seed-v1/8fab9f89f5242c6819f2...


Inspect row counts against Bronze, voucher balances, invoice-to-GL checks, calculated AR and POS controls. PASS and FAIL are evaluated comparisons. NOT_COVERED means the selected Bronze data lacks comparable evidence for that check. Resolve failed controls and error-level exceptions before publishing. Review other exceptions with the source manifest and record key available in lineage.

In [2]:
controls = pd.DataFrame(model.controls)
display(controls)
if not controls.empty:
    display(controls.groupby("status").size().rename("count"))
display(pd.DataFrame(model.calculated_ar))
display(pd.DataFrame(model.exceptions))
display(pd.DataFrame(model.lineage).head(20))
display(pd.DataFrame(list(model.coverage.items()), columns=["area", "coverage"]))

,rule,id,expected,actual,status,detail
0,order_headers_to_bronze,parent,6,6,PASS,
1,order_lines_to_bronze,parent,6,6,PASS,
2,order_headers_to_bronze,dealer,6,6,PASS,
3,order_lines_to_bronze,dealer,6,6,PASS,
4,records_to_bronze,billing_document,8,8,PASS,
5,records_to_bronze,billing_document_line,5,5,PASS,
6,records_to_bronze,payment,2,2,PASS,
7,records_to_bronze,gl_posting_line,32,32,PASS,
8,records_to_bronze,receivables_snapshot,2,2,PASS,
9,voucher_balance,"[""dealer"",""dealer voucher"",""D-BRONZE-R2-CREDIT...",10000,10000,PASS,


status
NOT_COVERED     3
PASS           34
Name: count, dtype: int64

,billing_document_id,legal_entity_id,cutoff,currency,open_amount,days_overdue,aging_bucket,coverage
0,"[""parent"",""ACC-SINV-2026-00001""]",parent,2026-12-16,INR,35000.0,13,1-30,linked fixture documents
1,"[""parent"",""ACC-SINV-2026-00002""]",parent,2026-12-16,INR,35000.0,11,1-30,linked fixture documents
2,"[""parent"",""ACC-SINV-2026-00004""]",parent,2026-12-16,INR,10000.0,31,31-60,linked fixture documents
3,"[""dealer"",""D-BRONZE-R2-INVOICE""]",dealer,2026-12-16,INR,None,1,1-30,incomplete credit or balance evidence
4,"[""dealer"",""D-I1""]",dealer,2026-12-16,INR,30000,11,1-30,linked fixture documents


,rule,table,id,severity,detail
0,unresolved_reference,billing_document,"[""dealer"",""D-BRONZE-R2-CREDIT""]",review,"{'role': 'original_invoice', 'target': 'billin..."
1,bank_account_not_in_feed,bank_statement_line,"[""parent"",""P-B1""]",review,P-P1
2,bank_account_not_in_feed,bank_statement_line,"[""dealer"",""D-B1""]",review,D-P1


,table,id,source_record_id,bronze_key
0,legal_entity,parent,OTC Parent Demo,batch/erpnext/manual__2026-11-21T15%3A27%3A44....
1,legal_entity,dealer,dealer_legacy,batch/dealer-sql/manual__2026-11-21T15%3A28%3A...
2,source_customer,"[""parent"",""OTC P-C1 Metro Industrial Services""]",OTC P-C1 Metro Industrial Services,batch/erpnext/manual__2026-11-21T15%3A27%3A44....
3,source_customer,"[""parent"",""OTC P-C2 Pioneer Engineering""]",OTC P-C2 Pioneer Engineering,batch/erpnext/manual__2026-11-21T15%3A27%3A44....
4,source_customer,"[""parent"",""OTC P-C3 Apex Industrial Supply""]",OTC P-C3 Apex Industrial Supply,batch/erpnext/manual__2026-11-21T15%3A27%3A44....
5,source_customer,"[""parent"",""OTC P-C4 Summit Process Systems""]",OTC P-C4 Summit Process Systems,batch/erpnext/manual__2026-11-21T15%3A27%3A44....
6,source_customer,"[""parent"",""OTC P-C5 Harbor Utilities""]",OTC P-C5 Harbor Utilities,batch/erpnext/manual__2026-11-21T15%3A27%3A44....
7,source_customer,"[""dealer"",""D-C1""]",D-C1,batch/dealer-sql/manual__2026-11-21T15%3A28%3A...
8,source_customer,"[""dealer"",""D-C2""]",D-C2,batch/dealer-sql/manual__2026-11-21T15%3A28%3A...
9,source_customer,"[""dealer"",""D-C3""]",D-C3,batch/dealer-sql/manual__2026-11-21T15%3A28%3A...


,area,coverage
0,inventory_movement_line,ERPNext Stock Entry legs only. Shipment and re...
1,receivables_snapshot,Dealer published balances only. Parent ERP cur...
2,billing_document_line,ERPNext item detail only. Dealer CSV has docum...
3,storefront_orders,Dealer export headers retained in Bronze. Deal...
4,pos_tender,Parent tender method only. Dealer export captu...
5,bank_match_allocation,Bank matches remain candidates pending account...
6,bank_account,Verified account records require explicit conf...
7,price_list,Catalog and agreements are outside the current...
8,source_user,Source user master is outside the current Bron...


The MinIO target writes Delta tables under one release prefix. Each table has a Delta transaction log and Parquet data files when it contains data. The release manifest is committed last and pins each table version. Paste the exact URI printed here into Gold settings. The local target remains a legacy JSONL export.

In [3]:
from silver_notebook import publish, publish_to_minio

PUBLISH_TARGET = "minio"  # Set to "minio" after reviewing the controls.
SILVER_BUCKET = "otc-silver"

if PUBLISH_TARGET == "minio":
    print(publish_to_minio(model, bucket=SILVER_BUCKET))
elif PUBLISH_TARGET == "local":
    print((publish(model).resolve() / "manifest.json").as_posix())
elif PUBLISH_TARGET == "preview":
    print('Preview complete. Set PUBLISH_TARGET = "minio" to publish the validated release.')
else:
    raise ValueError('Use "preview", "minio", or "local".')

s3://otc-silver/releases/409a707f49d90b112334f8bb4bfd27423a7f3f48f7039eecdff871dddee256e5/manifest.json


In [4]:
import inspect
import silver_notebook

print("Loaded publisher from:", silver_notebook.__file__)

if "storage_options" not in inspect.signature(
    silver_notebook.publish_to_minio
).parameters:
    raise RuntimeError("This kernel loaded the old JSONL publisher. Check the path above.")

manifest_uri = silver_notebook.publish_to_minio(model, bucket="otc-silver")
print("Open this exact release in MinIO:", manifest_uri)

Loaded publisher from: f:\otc-data-platform\silver\silver_notebook.py
Open this exact release in MinIO: s3://otc-silver/releases/409a707f49d90b112334f8bb4bfd27423a7f3f48f7039eecdff871dddee256e5/manifest.json
